# 4. Prepare usable four-month samples

The main workflow is **04 windows → 05 features → 06 training**. `analysis_mode = "pilot"` permits unmet collection budgets; `"strict"` requires those budgets. Both modes keep the same timing, completeness and parcel-separation checks. No imagery is downloaded here. Previous archives, splits, models and immutable bundles remain preserved.


## 1. Setup and configuration

Use the same collection and analysis mode in 04–06. Pilot is the default for testing the currently available data. Optional formal production exports keep their separate review/freeze requirements.


In [ ]:
from pathlib import Path
import json
import os
import sys
repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
os.chdir(repository_root)
from crop_pipeline import analysis_workflow, progress as pipeline_progress
from crop_pipeline.notebook_display import show_table, show_mapping, show_items
from crop_pipeline.sources import adaptive, annual, pilot_sources
from crop_pipeline.windows import seasons as season_workflow
from crop_pipeline.common import save_json
from IPython.display import display

analysis_mode = "pilot"  # pilot: report unmet budgets; strict: require collection targets
analysis_workflow.validate_mode(analysis_mode)
local_settings = json.loads(Path("settings.local.json").read_text()) if Path("settings.local.json").exists() else {}
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
source_run_name = local_settings.get("analysis_collection", adaptive.COLLECTION_NAME)
source_manifest_paths, analysis_run_name = pilot_sources.resolve_collection(report_folder, source_run_name)
analysis_folder = report_folder.parent / "analysis" / analysis_run_name
season_folder = analysis_folder / "seasons"
workflow_folder = season_folder / "workflow" / analysis_mode
print("Collection:", source_run_name, "| Mode:", analysis_mode)
print("Workflow folder:", workflow_folder)

from crop_pipeline.windows.workflow import WindowAnalysis
minimum_valid_fraction = 0.8
show_examples = True
export_formal_handoffs = False  # Advanced reviewed/frozen production inputs only
run_development_experiments = False  # Advanced diagnostics; never use reserved test groups
stages = ["Load completed imagery", "Audit saved imagery", "Select usable windows", "Save window handoff", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook("04_four_month_windows", stages, report_folder.parent / "progress", {"analysis": analysis_run_name, "mode": analysis_mode})
# Invalidate only current pointers; previous immutable bundles stay intact.
analysis_workflow.write_stage_pointer(workflow_folder / "window_stage.json", status="pending", reason="Notebook 04 has not finished this assessment")
analysis_workflow.write_stage_pointer(workflow_folder / "feature_stage.json", status="pending", reason="Run notebook 05 after the current window assessment")


## 2. Load completed imagery

Read verified source handoffs. Finish launched source batches first; partial source coverage is allowed. Tile counts are not parcel counts.


In [ ]:
pipeline_progress.begin_stage("Load completed imagery")
sources = pilot_sources.load_sources(source_manifest_paths, allow_existing_version_3=False, require_all=True)
show_items([source["source_name"] for source in sources], title="Sources")
analysis = WindowAnalysis(sources, source_manifest_paths, analysis_folder, report_folder,
    source_run_name, minimum_valid_fraction, run_development_experiments)
pipeline_progress.end_stage()


## 3. Audit imagery and timelines

Check bands, masks, grids, nodata and clear-observation counts. Scale reflectance by 0.0001; never scale counts. Keep the 80% coverage rule. Use saved actual-date summaries for timing, with composite-only evidence remaining provisional.


In [ ]:
pipeline_progress.begin_stage("Audit saved imagery")
with pipeline_progress.quiet_output():
    analysis.audit()
from collections import Counter
print("Tiles audited:", len(analysis.regions))
show_mapping(dict(Counter(row["status"] for row in analysis.quality_rows)), title="Half-month quality outcomes", key_label="Outcome", value_label="Intervals")
pipeline_progress.end_stage()


## 4. Select usable windows

Assess supported growth cycles, then retain the earliest timing-qualified window with complete usable pixels. Each window spans four calendar months and eight half-month bins. Missing values are excluded, not imputed. Keep at most 20 pixels per parcel and preserve locked partitions. Save exclusions and measured deficits.


In [ ]:
pipeline_progress.begin_stage("Select usable windows")
with pipeline_progress.quiet_output():
    analysis.select()
print("Reference-supported usable pixels:", len(analysis.feature_rows))
print("Timing-qualified estimated usable pixels:", len(analysis.estimated_feature_rows))
show_table(analysis.usable_selection_summary, title="Usable samples and remaining collection needs")
pipeline_progress.end_stage()


## 5. Save the window handoff

Export a hash-verified, input-keyed bundle for notebook 05. Both modes require all three crops, at least five usable source-parcel groups per crop, and each crop in every locked partition. This is an execution minimum, not evidence of accuracy. A blocked assessment cannot silently reuse an older handoff.


In [ ]:
pipeline_progress.begin_stage("Save window handoff")
coverage_summary = analysis.coverage()
coverage_summary.update(
    minimum_valid_fraction=minimum_valid_fraction,
    window_recipe={"detector_parameters": analysis.detector_settings,
        "automatic_timing_policy": analysis.configuration["automatic_timing_policy"],
        "cycle_selection_policy": "earliest_complete", "minimum_valid_fraction": minimum_valid_fraction,
        "window": "four_calendar_months", "composites": 8,
        "missing_values": "exclude_no_imputation", "reflectance_scale": 0.0001},
)
window_handoff = None
readiness_reason = None
try:
    if source_run_name != adaptive.COLLECTION_NAME:
        raise ValueError("The main pilot/strict handoff requires adaptive_training and its locked split; legacy archives remain separate")
    collection_readiness = analysis_workflow.require_collection_readiness(report_folder, analysis_mode)
    coverage_summary.update(collection_readiness=collection_readiness, targets_met=collection_readiness["targets_met"])
    coverage_path = workflow_folder / "assessment.json"
    save_json(coverage_path, coverage_summary)
    window_handoff = analysis_workflow.export_window_handoff(
        [analysis.feature_folder / "complete_features.csv", analysis.estimated_feature_folder / "complete_features.csv"],
        workflow_folder / "windows", collection_readiness["split_path"], coverage_path,
        analysis_mode=analysis_mode, collection_name=source_run_name,
        source_inputs_path=analysis_folder / "source_inputs.json",
    )
except (ValueError, RuntimeError) as error:
    readiness_reason = str(error)
    analysis_workflow.write_stage_pointer(workflow_folder / "window_stage.json", status="pending", reason=readiness_reason)
    save_json(workflow_folder / "assessment.json", {**coverage_summary, "readiness": "pending", "reason": readiness_reason})
else:
    analysis_workflow.write_stage_pointer(workflow_folder / "window_stage.json", window_handoff)
if export_formal_handoffs:
    with pipeline_progress.quiet_output():
        analysis.export_formal()
    print("Advanced formal handoffs retain their review/freeze requirements:", season_folder / "training_input")
pipeline_progress.end_stage("completed" if window_handoff else "pending")


## 6. Summary and next action

Read usable pixel/parcel counts, unmet budgets and readiness below. Estimated windows remain distinct from reference-supported windows. Optional examples explain saved results; they do not establish crop accuracy. See [workflow](../docs/workflow.md).


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Window handoff:", window_handoff or "pending")
print("Assessment:", workflow_folder / "assessment.json")
feedback_path = adaptive.root(report_folder) / "feedback.json"
if feedback_path.exists():
    feedback = json.loads(feedback_path.read_text())
    print("Collection targets met:", feedback["targets_met"])
    if analysis_mode == "pilot" and not feedback["targets_met"]:
        print("Pilot mode reports these deficits; it does not waive quality or split checks.")
if window_handoff:
    rows, _, _, _ = analysis_workflow.load_window_handoff(window_handoff)
    from crop_pipeline.training.exploratory import split_summary
    show_table(split_summary(rows), title="Usable samples by crop and partition")
    print("Next: run 05_feature_engineering.ipynb with the same analysis_mode.")
else:
    print("Blocked:", readiness_reason)
    print("Next: inspect exclusions and split coverage; collect eligible parcels for missing classes/partitions. Do not relax quality checks or move locked parcels.")
if show_examples and window_handoff and all(any(row["crop"] == crop for row in analysis.estimated_feature_rows) for crop in annual.TARGET_CROPS):
    from crop_pipeline.visualization import preprocessing as preprocessing_visuals
    import matplotlib.pyplot as plt
    with pipeline_progress.quiet_output():
        examples, example_folder = preprocessing_visuals.generate_examples(report_folder, analysis_run_name)
    for crop, figures in examples.items():
        print(crop, "— saved timing example")
        # Keep one compact preview; all reference/mask/composite figures remain saved.
        display(next(iter(figures.values())))
        for figure in figures.values():
            plt.close(figure)
    print("Saved preprocessing figures:", example_folder)
if show_examples and window_handoff and not all(any(row["crop"] == crop for row in analysis.estimated_feature_rows) for crop in annual.TARGET_CROPS):
    print("Estimated-window previews need a retained estimated sample per crop; the verified window handoff is ready independently.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
